# Lab 08: Set Up Microsoft Foundry Project

> **Source:** Microsoft Learning -- [mslearn-genaiops](https://github.com/MicrosoftLearning/mslearn-genaiops), `docs/01-infrastructure-setup.md`
> **License:** MIT

## Objectives

By the end of this lab you will:

1. Clone the **mslearn-genaiops** template repository
2. Provision Azure infrastructure with **`azd up`**
3. Generate a `.env` file with connection strings and keys
4. Deploy the **trail-guide agent** to Azure AI Foundry
5. Test the agent interactively

| Estimated Time | Estimated Cost |
|---|---|
| ~20 minutes | ~$1--2 |

## Architecture

![Architecture](architecture.png)

### Foundry Hierarchy

Azure AI Foundry organises resources in a three-level hierarchy:

| Level | Purpose |
|---|---|
| **Hub** | Shared governance boundary -- networking, identity, policies. One hub per team or department. |
| **Project** | Working environment inside a hub. Contains model deployments, agents, datasets, and evaluation runs. |
| **Agent / Deployment** | Individual resources inside a project. An *agent* is a persistent, stateful LLM configuration; a *deployment* is a model endpoint (e.g., GPT-4.1 Global Standard). |

Everything we provision in this lab lives inside **one Hub and one Project**.

## Step 1: Clone the Template Repository

Clone the Microsoft Learning repository that contains the Bicep templates, agent source code, and evaluation datasets:

```bash
git clone https://github.com/MicrosoftLearning/mslearn-genaiops.git
cd mslearn-genaiops
```

The repository structure includes:
- `infra/` -- Bicep templates for Azure resources
- `src/api/` -- Agent source code (trail-guide)
- `src/evaluators/` -- Automated evaluation scripts
- `data/` -- Evaluation datasets

## Step 2: Authenticate

Log in to both the Azure CLI (`az`) and the Azure Developer CLI (`azd`). These are **separate tools** with separate auth sessions:

```bash
az login
azd auth login
```

- **`az`** is the general-purpose Azure CLI for managing resources.
- **`azd`** is the Azure Developer CLI for template-based provisioning and deployment.

## Step 3: Provision with `azd up`

`azd up` is a single command that **provisions infrastructure** (via Bicep) and **deploys application code**. Under the hood it runs `azd provision` + `azd deploy`.

```bash
azd env new ai300-genaiops
azd up
```

You will be prompted to select:
- **Azure subscription**
- **Azure region** (choose one that supports GPT-4.1, e.g., `eastus2`)

This creates:
- An **AI Foundry Hub** and **Project**
- **GPT-4.1** and **GPT-4.1-mini** deployments (Global Standard SKU)
- **Application Insights** and **Log Analytics** for monitoring
- Required role assignments and networking

> **Exam Tip:** `azd` is the **Azure Developer CLI** -- separate from the `az` CLI. The command `azd up` = `azd provision` + `azd deploy`. It uses **Bicep-based Infrastructure as Code (IaC)** under the hood. Know the difference between `az` and `azd` for the exam.

## Step 4: Generate `.env` File

After provisioning, export all environment variables (connection strings, endpoint URLs, API keys) to a local `.env` file:

```bash
azd env get-values > .env
```

This file contains:
- `AZURE_AI_PROJECT_ENDPOINT` -- the Foundry project endpoint
- `AZURE_OPENAI_DEPLOYMENT_NAME` -- the model deployment name
- `APPLICATIONINSIGHTS_CONNECTION_STRING` -- for monitoring (Lab 12)
- Additional keys for evaluation and agent configuration

> **Important:** The `.env` file contains secrets. It is already listed in `.gitignore` -- never commit it to source control.

## Step 5: Create Virtual Environment

Set up an isolated Python environment and install dependencies:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

Key packages installed:
- `azure-ai-projects` -- AIProjectClient for agents and evaluations
- `azure-identity` -- DefaultAzureCredential for authentication
- `azure-monitor-opentelemetry` -- telemetry exporter (Lab 12)
- `python-dotenv` -- loads `.env` into `os.environ`

## Step 6: Deploy the Trail-Guide Agent

An **agent** in Azure AI Foundry is a **persistent, stateful LLM configuration**. Unlike a raw model deployment, an agent has:
- A **system prompt** (instructions)
- A **model binding** (e.g., GPT-4.1)
- Optional **tools** (code interpreter, file search, functions)
- A **thread** model for multi-turn conversations

Deploy the trail-guide agent:

```bash
python src/api/trail_guide_agent.py
```

This script uses `AIProjectClient.agents.create_agent()` to register the agent in your Foundry project. The agent ID is printed to the console and saved for use in later labs.

## Step 7: Test the Agent

Run the interactive test script to verify your agent responds correctly:

```bash
python src/api/interact_with_agent.py
```

Try asking:
- *"What gear do I need for a day hike?"*
- *"Help me plan a 3-day backpacking trip in Yosemite."*
- *"What should I do if I encounter a bear on the trail?"*

The script creates a **thread**, sends your message, runs the agent, and prints the response. Each conversation gets its own thread for context isolation.

## Step 8: Verify in Azure AI Foundry Portal

Navigate to [ai.azure.com](https://ai.azure.com) and verify the provisioned resources:

1. **Hub** -- find your AI Foundry Hub in the left navigation. Confirm it is in the region you selected.
2. **Project** -- open the project inside the hub. This is where all agents, deployments, and evaluations live.
3. **Model Deployments** -- under *Deployments*, confirm both **GPT-4.1** (Global Standard) and **GPT-4.1-mini** (Global Standard) are listed and active.
4. **Agent** -- under *Agents*, find the **trail-guide** agent you just deployed. Click into it to see the system prompt and model binding.

> **Note:** The portal may take 1--2 minutes to reflect newly deployed agents.

## Key Takeaways

| Concept | Detail |
|---|---|
| **Foundry Hierarchy** | Hub > Project > Agent/Deployment. Hub is the governance boundary; Project is the working environment. |
| **`azd` for IaC** | Azure Developer CLI provisions infrastructure from Bicep templates. `azd up` = provision + deploy in one command. |
| **Global Standard Deployments** | Pay-per-token, no reserved capacity. Best for development and variable workloads. |
| **Agents as Persistent Resources** | Unlike raw API calls, agents are registered in Foundry with their own ID, instructions, and thread model. |
| **`.env` for Configuration** | `azd env get-values > .env` exports all connection strings and keys. Never commit this file. |

---

**Next:** [Lab 09 -- Prompt Versioning](../lab09-prompt-versioning/lab09-prompt-versioning.ipynb) -- evolve the trail-guide prompt across V1/V2/V3.